# Hover Logix YOLOv8 v2: train, test, export for Android
Classes: 0 pallet, 1 pallet_label, 2 location_label, 3 beam, 4 upright, 5 empty_location, 6 pallet_base.
Runtime: GPU (T4). Put `hoverlogix_dataset_v2.zip` in Google Drive folder `HoverLogix`. Run all cells.

In [ ]:
# 1. Setup
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U ultralytics
import ultralytics; ultralytics.checks()
DRIVE='/content/drive/MyDrive/HoverLogix'
MODEL_SIZE='n'      # 'n' for RC 2 Plus + S24 (fast); 's' is more accurate but ~3x slower
IMGSZ=640

In [ ]:
# 2. Data
import os, glob, zipfile, shutil
ZIP = f'{DRIVE}/hoverlogix_dataset_v2.zip'
if not os.path.exists(ZIP):   # find it anywhere in My Drive and save results next to it
    hits = glob.glob('/content/drive/MyDrive/**/hoverlogix_dataset_v2.zip', recursive=True)
    assert hits, 'Put hoverlogix_dataset_v2.zip in Google Drive'
    ZIP = hits[0]; DRIVE = os.path.dirname(ZIP)
print('dataset:', ZIP, '| results go to:', DRIVE)
if not os.path.exists('/content/data/data.yaml'):
    shutil.copy(ZIP, '/content/ds.zip')
    zipfile.ZipFile('/content/ds.zip').extractall('/content/data')
for s in ('train','val','test'):
    print(s, len(glob.glob(f'/content/data/images/{s}/*.jpg')), 'images')
print(open('/content/data/data.yaml').read())

In [ ]:
# 3. Stage 1 training
from ultralytics import YOLO
m = YOLO(f'yolov8{MODEL_SIZE}.pt')
m.train(data='/content/data/data.yaml', imgsz=IMGSZ, epochs=100, patience=25, batch=32, workers=4,
        cos_lr=True, close_mosaic=15, degrees=3, fliplr=0.5, project='/content/runs', name='stage1', exist_ok=True, plots=True)
S1='/content/runs/stage1/weights/best.pt'

In [ ]:
# 4. Fill in classes each source never labelled (self-training)
# Public pallet photos only mark pallet_base; DroneIMS photos never mark pallet_base.
# Add confident stage-1 predictions for the missing classes so the model is not taught they are background.
import numpy as np
from ultralytics import YOLO
t = YOLO(S1)
MISSING = {'pd_': [0,1,2,3,4], 'wod_': [1,2,3,4,6], 'dims_': [6]}
CONF = 0.55
def iou(a,b):
    iw=min(a[2],b[2])-max(a[0],b[0]); ih=min(a[3],b[3])-max(a[1],b[1])
    if iw<=0 or ih<=0: return 0
    I=iw*ih; return I/((a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-I)
added={}
for split in ('train','val'):
    ims = sorted(glob.glob(f'/content/data/images/{split}/*.jpg'))
    for i in range(0, len(ims), 64):
        batch = ims[i:i+64]
        for r in t.predict(batch, imgsz=IMGSZ, conf=CONF, verbose=False, stream=True):
            name = os.path.basename(r.path)
            pref = next((p for p in MISSING if name.startswith(p)), None)
            if not pref: continue
            lp = r.path.replace('/images/','/labels/')[:-4]+'.txt'
            old = [l.split() for l in open(lp)] if os.path.exists(lp) else []
            oldb = [(int(v[0]), [float(v[1])-float(v[3])/2, float(v[2])-float(v[4])/2, float(v[1])+float(v[3])/2, float(v[2])+float(v[4])/2]) for v in old if len(v)==5]
            new=[]
            for c,b in zip(r.boxes.cls.cpu().numpy().astype(int), r.boxes.xyxyn.cpu().numpy()):
                if c not in MISSING[pref]: continue
                if any(oc==c and iou(ob,b)>0.5 for oc,ob in oldb): continue
                new.append(f"{c} {(b[0]+b[2])/2:.6f} {(b[1]+b[3])/2:.6f} {b[2]-b[0]:.6f} {b[3]-b[1]:.6f}")
                added[c]=added.get(c,0)+1
            if new:
                txt = open(lp).read() if os.path.exists(lp) else ''
                if txt and not txt.endswith('\n'): txt += '\n'
                open(lp,'w').write(txt + '\n'.join(new) + '\n')
print('pseudo-labels added per class:', dict(sorted(added.items())))
for d in glob.glob('/content/data/labels/*.cache'): os.remove(d)

In [ ]:
# 5. Stage 2 training on the completed labels
m = YOLO(S1)
m.train(data='/content/data/data.yaml', imgsz=IMGSZ, epochs=60, patience=20, batch=32, workers=4, lr0=0.003,
        cos_lr=True, close_mosaic=10, degrees=3, project='/content/runs', name='stage2', exist_ok=True, plots=True)
BEST='/content/runs/stage2/weights/best.pt'

In [ ]:
# 6. Test on held-out photos (never used in training)
import yaml
m = YOLO(BEST)
res = {}
for name, prefix in [('all','') , ('droneims_hand_labelled','dims_'), ('public_pallet_base','pd_')]:
    lst = [p for p in sorted(glob.glob('/content/data/images/test/*.jpg')) if os.path.basename(p).startswith(prefix)]
    open(f'/content/data/test_{name}.txt','w').write('\n'.join(lst))
    y = yaml.safe_load(open('/content/data/data.yaml')); y['test'] = f'/content/data/test_{name}.txt'
    yaml.safe_dump(y, open(f'/content/data/test_{name}.yaml','w'))
    v = m.val(data=f'/content/data/test_{name}.yaml', split='test', imgsz=IMGSZ, batch=16, plots=True,
              project='/content/runs', name=f'test_{name}', exist_ok=True, verbose=True)
    res[name] = {'images': len(lst), 'mAP50': round(float(v.box.map50),3), 'mAP50_95': round(float(v.box.map),3),
                 'per_class_mAP50': {m.names[int(c)]: round(float(a),3) for c,a in zip(v.box.ap_class_index, v.box.ap50)}}
import json; print(json.dumps(res, indent=1))
json.dump(res, open('/content/runs/test_results.json','w'), indent=1)

In [ ]:
# 7. Export for Android (LiteRT / TFLite) + ONNX
EXP='/content/export'; os.makedirs(EXP, exist_ok=True)
m = YOLO(BEST)
p = m.export(format='tflite', imgsz=IMGSZ, half=True)          # float16: best accuracy, GPU delegate
f16 = shutil.copy(p, f'{EXP}/hoverlogix_v2_{IMGSZ}_float16.tflite')
p = YOLO(BEST).export(format='tflite', imgsz=IMGSZ, int8=True, data='/content/data/data.yaml', fraction=0.2)  # int8: fastest on CPU
i8 = shutil.copy(p, f'{EXP}/hoverlogix_v2_{IMGSZ}_int8.tflite')
p = YOLO(BEST).export(format='onnx', imgsz=IMGSZ, simplify=True)
onx = shutil.copy(p, f'{EXP}/hoverlogix_v2_{IMGSZ}.onnx')
shutil.copy(BEST, f'{EXP}/hoverlogix_v2.pt')
print(os.listdir(EXP))

In [ ]:
# 8. Quick check: run the float16 TFLite on test photos and time it
import time, cv2
tfl = YOLO(f16, task='detect')
ims = sorted(glob.glob('/content/data/images/test/dims_*.jpg'))[:8] + sorted(glob.glob('/content/data/images/test/pd_*.jpg'))[:4]
t0=time.time(); rs = tfl.predict(ims, imgsz=IMGSZ, conf=0.3, verbose=False); dt=(time.time()-t0)/len(ims)
print(f'TFLite on Colab CPU: {dt*1000:.0f} ms/image')
os.makedirs('/content/runs/samples', exist_ok=True)
for r in rs: cv2.imwrite('/content/runs/samples/'+os.path.basename(r.path), r.plot())

In [ ]:
# 9. Save everything to Drive
import shutil, glob
out = f'{DRIVE}/models/v2'; os.makedirs(out, exist_ok=True)
for p in glob.glob(f'{EXP}/*') + ['/content/runs/test_results.json']:
    shutil.copy(p, out)
shutil.make_archive(f'{DRIVE}/hoverlogix_model_v2', 'zip', '/content/runs')
print(os.listdir(out))